# Pre-trade

In [2]:
# Step 2: Import Python Packages: Pandas, Numpy, and Statistics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
from scipy.optimize import minimize
import os

In [3]:
# Step 3: Set I/O Folders
InputPath = 'inputFolder'

os.makedirs('outputFolder', exist_ok=True)
OutputPath = 'outputFolder'

In [4]:
# Step 4: Set MI Parameters - Calculated from Non-Linear Regression Analysis
a1=883.58722
a2=0.35408
a3=0.755684
a4=0.826155
b1=0.963532
MIParams = [a1, a2, a3, a4, b1]
print(MIParams)

[883.58722, 0.35408, 0.755684, 0.826155, 0.963532]


In [ ]:
# Step 5: User Defined TCA Pre-Trade Functions

# Develop Python TCA Pre-Trade functions to calculate:
# Market Impact (MI)
# Timing Risk (TR)
# Price Appreciation (PA)
# POV Rate to Trade Time (PovToTime)
# Trade Time to POV Rate (TimeToPOV)
# Expected Cost (ExpCost)
# Expected Execution Price (ExpPrice)

class PreTradeTCA:
    @staticmethod
    def MarketImpact(Size, POV, Volatility):
        """ Calculate Market Impact (MI) in bps"""
        Istar = a1 * (Size ** a2) * (Volatility ** a3)
        MI = Istar * (b1 * POV ** a4 + (1 - b1))
        return MI
    
    @staticmethod
    def TimingRisk(Size, POV, Volatility):
        """Calculates Timing Risk (TR) in bps """
        TR = Volatility * np.sqrt((1/3) * (1/250) * Size * ((1 - POV)/POV)) * 10000
        return TR
    
    @staticmethod
    def PriceAppreciation(Side, AlphaBp, Size, POV):
        """Calculates Price Appreciation (PA) in bps """
        PA = Side * 0.5 * AlphaBp * Size * ((1 - POV)/POV)
        return PA
    
    @staticmethod
    def PovToTime(Size, POV):
        """Converts POV rate to Trade Time """
        return Size * ((1 - POV) / POV)
    
    def TimeToPOV(Self, Size, TradeTime):
        """Converts Trade Time to POV rate """
        return Size / (Size + TradeTime)
    
    @staticmethod
    def ExpectedCost(MI, PA):
        """Calculates Expected Cost"""
        return MI + PA
    
    @staticmethod
    def ExpectedPrice(P0, Side, MI, PA):
        """Calculates Expected Execution Price """
        return P0 * (1 + (Side * (MI + PA) / 10000))

In [6]:
tca = PreTradeTCA()

# --------------------------------------
# Homework 7
# --------------------------------------

## Pre-Trade Calculations

### Question 1

In [7]:
Side = 1
Size = 0.10
Volatility = 0.35
AlphaBp = 30
POV = 0.20

MI = tca.MarketImpact(Size=Size, POV=POV, Volatility=Volatility)
TR = tca.TimingRisk(Size=Size, POV=POV, Volatility=Volatility)
PA = tca.PriceAppreciation(Side=Side, AlphaBp=AlphaBp, Size=Size, POV=POV)
PovToTime = tca.PovToTime(Size=Size, POV=POV)

print(f'Market Impact: {MI:.4f} bps')
print(f'Timing Risk: {TR:.4f} bps')
print(f'Price Appreciation: {PA:.4f} bps')
print(f'POV : {POV * 100:.4f}%')
print(f'Trade Time: {PovToTime:.4f}')


Market Impact: 51.5353 bps
Timing Risk: 80.8290 bps
Price Appreciation: 6.0000 bps
POV : 20.0000%
Trade Time: 0.4000


### Question 2

In [8]:
Side = -1
Size = 125_000 / 1_000_000
Volatility = 0.40
AlphaBp = 10
POV = 0.10

MI = tca.MarketImpact(Size=Size, POV=POV, Volatility=Volatility)
TR = tca.TimingRisk(Size=Size, POV=POV, Volatility=Volatility)
PA = tca.PriceAppreciation(Side=Side, AlphaBp=AlphaBp, Size=Size, POV=POV)
PovToTime = tca.PovToTime(Size=Size, POV=POV)

print(f'Market Impact: {MI:.4f} bps')
print(f'Timing Risk: {TR:.4f} bps')
print(f'Price Appreciation: {PA:.4f} bps')
print(f'POV : {POV * 100:.4f}%')
print(f'Trade Time: {PovToTime:.4f}')

Market Impact: 38.1634 bps
Timing Risk: 154.9193 bps
Price Appreciation: -5.6250 bps
POV : 10.0000%
Trade Time: 1.1250


## Expected Cost and Expected Price

### Question 3

In [9]:
Side = 1
Size = 0.05
Volatility = 0.25
AlphaBp = 10
Price = 75
TradeTime = 1

POV = tca.TimeToPOV(Size=Size, TradeTime=TradeTime)
TradeTime = tca.PovToTime(Size=Size, POV=POV)
MI = tca.MarketImpact(Size=Size, POV=POV, Volatility=Volatility)
TR = tca.TimingRisk(Size=Size, POV=POV, Volatility=Volatility)
PA = tca.PriceAppreciation(Side=Side, AlphaBp=AlphaBp, Size=Size, POV=POV)
ExpectedCost = tca.ExpectedCost(MI=MI, PA=PA)
ExpectedPrice = tca.ExpectedPrice(P0=Price, Side=Side, MI=MI, PA=PA)

print(f'Market Impact: {MI:.4f} bps')
print(f'Timing Risk: {TR:.4f} bps')
print(f'Price Appreciation: {PA:.4f} bps')
print(f'POV : {POV * 100:.4f}%')
print(f'Trade Time: {TradeTime:.4f}')
print(f'Expected Cost: {ExpectedCost:.4f} bps')
print(f'Expected Price: {ExpectedPrice:.4f}')


Market Impact: 12.2715 bps
Timing Risk: 91.2871 bps
Price Appreciation: 5.0000 bps
POV : 4.7619%
Trade Time: 1.0000
Expected Cost: 17.2715 bps
Expected Price: 75.1295


### Question 4

In [10]:
Side = -1
Size = 75_000 / 650_000
Volatility = 0.30
AlphaBp = -10
Price = 250
TradeTime = 0.5

POV = tca.TimeToPOV(Size=Size, TradeTime=TradeTime)
TradeTime = tca.PovToTime(Size=Size, POV=POV)
MI = tca.MarketImpact(Size=Size, POV=POV, Volatility=Volatility)
TR = tca.TimingRisk(Size=Size, POV=POV, Volatility=Volatility)
PA = tca.PriceAppreciation(Side=Side, AlphaBp=AlphaBp, Size=Size, POV=POV)
ExpectedCost = tca.ExpectedCost(MI=MI, PA=PA)
ExpectedPrice = tca.ExpectedPrice(P0=Price, Side=Side, MI=MI, PA=PA)

print(f'Market Impact: {MI:.4f} bps')
print(f'Timing Risk: {TR:.4f} bps')
print(f'Price Appreciation: {PA:.4f} bps')
print(f'POV : {POV * 100:.4f}%')
print(f'Trade Time: {TradeTime:.4f}')
print(f'Expected Cost: {ExpectedCost:.4f} bps')
print(f'Expected Price: {ExpectedPrice:.5f}')

Market Impact: 46.0606 bps
Timing Risk: 77.4597 bps
Price Appreciation: 2.5000 bps
POV : 18.7500%
Trade Time: 0.5000
Expected Cost: 48.5606 bps
Expected Price: 248.78598


### Question 5

In [11]:
Side = 1
Size = 115_250 / 800_000
Volatility = 0.50
AlphaBp = -5
Price = 100
TradeTime = 0.75

POV = tca.TimeToPOV(Size=Size, TradeTime=TradeTime)
TradeTime = tca.PovToTime(Size=Size, POV=POV)
MI = tca.MarketImpact(Size=Size, POV=POV, Volatility=Volatility)
TR = tca.TimingRisk(Size=Size, POV=POV, Volatility=Volatility)
PA = tca.PriceAppreciation(Side=Side, AlphaBp=AlphaBp, Size=Size, POV=POV)
ExpectedCost = tca.ExpectedCost(MI=MI, PA=PA)
ExpectedPrice = tca.ExpectedPrice(P0=Price, Side=Side, MI=MI, PA=PA)

print(f'Market Impact: {MI:.4f} bps')
print(f'Timing Risk: {TR:.4f} bps')
print(f'Price Appreciation: {PA:.4f} bps')
print(f'POV : {POV * 100:.4f}%')
print(f'Trade Time: {TradeTime:.4f}')
print(f'Expected Cost: {ExpectedCost:.4f} bps')
print(f'Expected Price: {ExpectedPrice:.5f}')

Market Impact: 65.8059 bps
Timing Risk: 158.1139 bps
Price Appreciation: -1.8750 bps
POV : 16.1132%
Trade Time: 0.7500
Expected Cost: 63.9309 bps
Expected Price: 100.63931


### Question 6

In [12]:
Side = 1
Size = 100_000 / 2_000_000
Volatility = 0.40
AlphaBp = 0
Price = 100
POV = 0.25

TradeTime = tca.PovToTime(Size=Size, POV=POV)
POV = tca.TimeToPOV(Size=Size, TradeTime=TradeTime)
MI = tca.MarketImpact(Size=Size, POV=POV, Volatility=Volatility)
TR = tca.TimingRisk(Size=Size, POV=POV, Volatility=Volatility)
PA = tca.PriceAppreciation(Side=Side, AlphaBp=AlphaBp, Size=Size, POV=POV)
ExpectedCost = tca.ExpectedCost(MI=MI, PA=PA)
ExpectedPrice = tca.ExpectedPrice(P0=Price, Side=Side, MI=MI, PA=PA)

print(f'Market Impact: {MI:.4f} bps')
print(f'Timing Risk: {TR:.4f} bps')
print(f'Price Appreciation: {PA:.4f} bps')
print(f'POV : {POV * 100:.4f}%')
print(f'Trade Time: {TradeTime:.4f}')
print(f'Expected Cost: {ExpectedCost:.4f} bps')
print(f'Expected Price: {ExpectedPrice:.5f}')

Market Impact: 52.4993 bps
Timing Risk: 56.5685 bps
Price Appreciation: 0.0000 bps
POV : 25.0000%
Trade Time: 0.1500
Expected Cost: 52.4993 bps
Expected Price: 100.52499


## Trade Time

### Question 7

In [13]:
Size = 0.25
POV = 0.10

TradeTime = tca.PovToTime(Size=Size, POV=POV)

print(f'Trade Time: {TradeTime:.4f}')

Trade Time: 2.2500


## POV Rate

### Question 8

In [14]:
Size = 0.20
TradeTime = 0.75

POV = tca.TimeToPOV(Size=Size, TradeTime=TradeTime)

print(f'POV : {POV * 100:.4f}%')

POV : 21.0526%


## Maximum Shares

### Question 9

In [62]:
Volatility = 0.50
ADV = 2_000_000
TradeTime = 1
TargetMI = 75
tolerance = 0.0001


for shares in range(ADV):
    Size = shares / ADV
    POV = tca.TimeToPOV(Size=Size, TradeTime=TradeTime)
    MI = tca.MarketImpact(Size=Size, POV=POV, Volatility=Volatility)

    if abs(MI - TargetMI) < tolerance:
        print(f'Shares: {shares:,}')
        print(f'Market Impact: {MI:.4f} bps')
        print(f'Size: {Size * 100:.4f}%')
        # print(f'POV : {POV * 100:.4f}%')
        break

# ------- Binary Search Implementation -------
# Binary search is significantly faster if ADV is large

# l,r = 0, ADV

# while l <= r:
#     mid = (l + r) // 2
#     Size = mid / ADV
#     POV = tca.TimeToPOV(Size=Size, TradeTime=TradeTime)
#     MI = tca.MarketImpact(Size=Size, POV=POV, Volatility=Volatility)

#     if abs(MI - TargetMI) < tolerance:
#         print(f'Shares: {mid:,}')
#         print(f'Market Impact: {MI:.4f} bps')
#         print(f'Size: {Size * 100:.4f}%')
#         # print(f'POV : {POV * 100:.4f}%')
#         break
#     elif MI < TargetMI:
#         l = mid + 1
#     else:
#         r = mid - 1

Shares: 396,099
Market Impact: 75.0000 bps
Size: 19.8049%


## Trade Time from AlphaBp

### Question 10

In [57]:
P0 = 65
AlphaBp = 100
targetPrice = 65.45

# Calculate the total return required to reach the target
priceChange  = ((targetPrice / P0) - 1)

# Solve for t: required_return = alpha_decimal * t
time_in_days = priceChange  / (AlphaBp / 10000)

print(f'Time in days: {time_in_days:.5f} days')

Time in days: 0.69231 days
